<a href="https://colab.research.google.com/github/lcbjrrr/AOO/blob/main/DB_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
con = sqlite3.connect("DB.db")

In [3]:
cur = con.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS "Students" (
                                         "student_id" INTEGER NOT NULL UNIQUE,
                                         "name" VARCHAR NOT NULL,
                                         "gender" VARCHAR,
                                         PRIMARY KEY("student_id")
);
""")
cur.execute("""
CREATE TABLE IF NOT EXISTS "Grades" (
                                       "grade_id" INTEGER NOT NULL UNIQUE,
                                       "grade" NUMERIC NOT NULL,
                                       "score" VARCHAR NOT NULL,
                                       "student_fk" INTEGER,
                                       PRIMARY KEY("grade_id"),
                                       FOREIGN KEY ("student_fk") REFERENCES "Students"("student_id")
                                           ON UPDATE NO ACTION ON DELETE CASCADE
);
""")
cur.execute("""
CREATE TABLE IF NOT EXISTS "Classes" (
                                        "class_id" INTEGER NOT NULL UNIQUE,
                                        "code" VARCHAR NOT NULL,
                                        "name" VARCHAR NOT NULL,
                                        PRIMARY KEY("class_id")
);
""")
cur.execute("""
CREATE TABLE IF NOT EXISTS "Students_Classes" (
                                                 "stundent_fk" INTEGER NOT NULL,
                                                 "classes_fk" INTEGER NOT NULL,
                                                 PRIMARY KEY("stundent_fk", "classes_fk"),
                                                 FOREIGN KEY ("stundent_fk") REFERENCES "Students"("student_id")
                                                     ON UPDATE NO ACTION ON DELETE NO ACTION,
                                                 FOREIGN KEY ("classes_fk") REFERENCES "Classes"("class_id")
                                                     ON UPDATE NO ACTION ON DELETE NO ACTION
);
""")
con.commit()

In [5]:
cur.execute("""
INSERT INTO "Students" ("student_id", "name", "gender") VALUES
                                                           (1, 'John', 'Male'),
                                                           (2, 'Mary', 'Female'),
                                                           (3, 'Jim', 'Male'),
                                                           (4, 'Tony', 'Male'),
                                                           (5, 'Kate', 'Female');
""")
con.commit()

In [6]:
res = cur.execute("SELECT * FROM Students")
res.fetchall()

[(1, 'John', 'Male'),
 (2, 'Mary', 'Female'),
 (3, 'Jim', 'Male'),
 (4, 'Tony', 'Male'),
 (5, 'Kate', 'Female')]

In [7]:
cur.execute("""
INSERT INTO "Grades" ("grade_id", "grade", "score", "student_fk") VALUES
-- John (Student 1)
(1, 70, 'C', 1),
(2, 80, 'B', 1),
-- Mary (Student 2)
(3, 80, 'B', 2),
(4, 80, 'B', 2),
-- Jim (Student 3)
(5, 80, 'B', 3),
(6, 90, 'A', 3),
-- Tony (Student 4)
(7, 88, 'B+', 4),
(8, 99, 'A+', 4),
-- Kate (Student 5)
(9, 77, 'C+', 5),
(10, 88, 'B+', 5);

""")
con.commit()

In [8]:
res = cur.execute("""
SELECT
   S.name,
   G.grade
FROM "Students" S, "Grades" G
WHERE 	S.student_id = G.student_fk
""")
res.fetchall()

[('John', 70),
 ('John', 80),
 ('Mary', 80),
 ('Mary', 80),
 ('Jim', 80),
 ('Jim', 90),
 ('Tony', 88),
 ('Tony', 99),
 ('Kate', 77),
 ('Kate', 88)]

In [9]:
res = cur.execute("""
SELECT
   S.name,
   AVG(G.grade) AS average_grade
FROM "Students" S, "Grades" G
WHERE 	S.student_id = G.student_fk
GROUP BY S.name;
""")
res.fetchall()

[('Jim', 85.0), ('John', 75.0), ('Kate', 82.5), ('Mary', 80.0), ('Tony', 93.5)]

In [10]:
cur.execute("""
INSERT INTO "Classes" ("class_id", "code", "name") VALUES
                                                      (1, 'CSC230', 'Intro OO'),
                                                      (2, 'CSC420', 'SW Collaboration');

""")

cur.execute("""
INSERT INTO "Students_Classes" ("stundent_fk", "classes_fk") VALUES
                                                                (1, 1), -- John -> CSC230
                                                                (2, 1), -- Mary -> CSC230
                                                                (3, 1), -- Jim  -> CSC230
                                                                (4, 2), -- Tony -> CSC420
                                                                (5, 2), -- Kate -> CSC420
                                                                (1, 2); -- John -> CSC420

""")
con.commit()







In [11]:
res = cur.execute("""
SELECT
   C.code AS Course_Code,
   COUNT(DISTINCT SC.stundent_fk) AS Student_Count
FROM "Classes" C, "Students_Classes" SC
WHERE C.class_id = SC.classes_fk
GROUP BY C.class_id
""")
res.fetchall()






[('CSC230', 3), ('CSC420', 3)]

In [12]:


res = cur.execute("""
SELECT
   C.code AS Course_Code,
   COUNT(DISTINCT SC.stundent_fk) AS Student_Count,
   AVG(G.grade) AS Average_Grade
FROM "Classes" C, "Students_Classes" SC, "Grades" G
WHERE C.class_id = SC.classes_fk
 AND SC.stundent_fk = G.student_fk
GROUP BY  C.code;
""")
res.fetchall()


[('CSC230', 3, 80.0), ('CSC420', 3, 83.66666666666667)]

In [13]:
res = cur.execute("""
SELECT
   S.name AS Student_Name,
   C.code AS Course_Code,
   AVG(G.grade) AS Average_Grade
FROM "Students" S, "Students_Classes" SC, "Classes" C, "Grades" G
WHERE S.student_id = SC.stundent_fk
 and SC.classes_fk = C.class_id
 and S.student_id = G.student_fk
GROUP BY S.student_id, C.class_id;
""")
res.fetchall()






[('John', 'CSC230', 75.0),
 ('John', 'CSC420', 75.0),
 ('Mary', 'CSC230', 80.0),
 ('Jim', 'CSC230', 85.0),
 ('Tony', 'CSC420', 93.5),
 ('Kate', 'CSC420', 82.5)]

![](https://pbs.twimg.com/media/HBm-Q6VXoAAR_tf?format=jpg&name=medium)